# Other organisms — mouse and pig showcase

This notebook provides a small **non-human** showcase to demonstrate that IDTrack is not limited to human.

## Rationale

The core IDTrack idea (Ensembl history as a time axis + snapshot-bounded mapping + explicit ambiguity) applies to any Ensembl-supported organism.
This experiment intentionally stays **within-species** (no ortholog mapping) and answers:

- Can we build/load the organism graph snapshot from the shared cache?
- Do we see the same 1→0 / 1→1 / 1→n outcome semantics?
- Does external matching (e.g. UniProt) preserve the same explicit ambiguity semantics?

This is a marketing appendix: it shows portability without diluting the main human-focused narrative.

Outputs:
- `idtrack-manuscript/figures/fig_other_organisms_outcomes.pdf`
- `idtrack-manuscript/tables/other_organisms_showcase_summary.csv`

Caching:
- Results are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/other_organisms/`.
- If caches are missing, the notebook computes them (graph loading can be memory-intensive).

Related:
- Cross-species harmonization into human is covered separately in `idtrack/docs/_notebooks/06_tutorial_humanization_mouse_pig_to_human.ipynb`.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    notebook_context,
    read_pickle,
    save_figure,
    write_pickle,
)

from idtrack_results import summarize_binned_conversion  # noqa: E402

ctx = notebook_context('other_organisms', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

# Demo settings (keep small; this is a *showcase*, not a benchmark)
N_SAMPLE_IDS = 200
STRATEGY = 'all'   # exposes ambiguity (1→n)
TO_RELEASE = 110   # choose a reasonable mid-range release for the demo

# Scenarios: staying on the Ensembl backbone vs external matching.
# Note: HGNC is human-specific; for non-human we focus on UniProt as a portable external target.
SCENARIOS = [
    {'label': 'Ensembl backbone', 'final_database': None},
    {'label': 'External match: UniProt', 'final_database': 'UniProtKB/Swiss-Prot'},
]

ORGANISMS = [
    ('mouse', 'ENSMUSG'),
    ('pig', 'ENSSSCG'),
]

RESULTS_PKL = CACHE_DIR / f"other_organisms_summary_n{N_SAMPLE_IDS}_to{TO_RELEASE}_strategy{STRATEGY}.pickle"

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import numpy as np
import pandas as pd

needs_recompute = True
if RESULTS_PKL.exists():
    df = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
    if isinstance(df, pd.DataFrame) and {'scenario', '1_to_1_tdm', '1_to_1_atm', '1_to_n_tdm', '1_to_n_atm'}.issubset(df.columns):
        needs_recompute = False

if needs_recompute:
    import idtrack

    rng = np.random.default_rng(0)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    def reservoir_sample(nodes, prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    rows = []
    for organism_alias, prefix in ORGANISMS:
        organism, latest = api.resolve_organism(organism_alias)
        snapshot = latest
        api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=False)

        sample = reservoir_sample(api.track.graph.nodes, prefix, N_SAMPLE_IDS)
        if not sample:
            print('No IDs found for', organism_alias, 'with prefix', prefix)
            continue

        to_release = min(int(TO_RELEASE), int(latest))

        for scenario in SCENARIOS:
            label = str(scenario['label'])
            final_db = scenario.get('final_database')

            matchings = api.convert_identifier_multiple(
                sample,
                from_release=int(latest),
                to_release=to_release,
                final_database=final_db,
                strategy=STRATEGY,
                verbose=True,
                pbar_prefix=f"{organism_alias} | {label}",
            )

            bins = api.classify_multiple_conversion(matchings)
            stats = summarize_binned_conversion(bins)

            rows.append(
                {
                    'organism': organism_alias,
                    'scenario': label,
                    'final_database': final_db if final_db is not None else 'Ensembl gene',
                    'snapshot_release': int(latest),
                    'to_release': int(to_release),
                    'n': int(stats['total']),
                    '1_to_0': int(stats['1_to_0']),
                    '1_to_1_tdm': int(stats['1_to_1_tdm']),
                    '1_to_1_atm': int(stats['1_to_1_atm']),
                    '1_to_n_tdm': int(stats['1_to_n_tdm']),
                    '1_to_n_atm': int(stats['1_to_n_atm']),
                    'changed_1_to_1': int(stats['changed_1_to_1']),
                    'changed_1_to_n': int(stats['changed_1_to_n']),
                }
            )

    df = pd.DataFrame(rows)
    write_pickle(df, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

df


In [ ]:
# -------------------- Plot outcome profiles --------------------

if df is None or df.empty:
    print('No results to plot.')
else:
    scenarios = list(dict.fromkeys(df['scenario'].tolist())) if 'scenario' in df.columns else ['(single)']
    nrows = len(scenarios)
    fig, axes = plt.subplots(nrows, 2, figsize=(12.0, 3.6 * nrows), constrained_layout=True)
    if nrows == 1:
        axes = np.array([axes])

    for r, scenario in enumerate(scenarios):
        sub = df[df.get('scenario', scenario) == scenario].copy() if 'scenario' in df.columns else df.copy()
        sub = sub.set_index('organism')

        n = sub['n'].replace(0, np.nan)
        frac = pd.DataFrame(
            {
                '1→0': sub['1_to_0'] / n,
                '1→1 TDM': sub.get('1_to_1_tdm', 0) / n,
                '1→1 ATM': sub.get('1_to_1_atm', 0) / n,
                '1→n TDM': sub.get('1_to_n_tdm', 0) / n,
                '1→n ATM': sub.get('1_to_n_atm', 0) / n,
            }
        )

        ax0 = axes[r, 0]
        frac.plot(
            kind='bar',
            stacked=True,
            ax=ax0,
            color=[
                MANUSCRIPT_COLORS['1→0'],
                MANUSCRIPT_COLORS['1→1 TDM'],
                MANUSCRIPT_COLORS['1→1 ATM'],
                MANUSCRIPT_COLORS['1→n TDM'],
                MANUSCRIPT_COLORS['1→n ATM'],
            ],
        )
        ax0.set_ylabel('Fraction of queries')
        ax0.set_xlabel('Organism')
        ax0.set_ylim(0, 1)
        ax0.set_title(f"Outcome profile — {scenario}")
        ax0.legend(loc='upper right', frameon=True)

        ax1 = axes[r, 1]
        changed = (sub.get('changed_1_to_1', 0) + sub.get('changed_1_to_n', 0)) / n
        ax1.bar(sub.index.tolist(), changed, color=MANUSCRIPT_COLORS['neutral'])
        ax1.set_ylim(0, 1)
        ax1.set_xlabel('Organism')
        ax1.set_ylabel('Fraction changed')
        ax1.set_title(f"Drift signal — {scenario}")

    written = save_figure(fig, 'fig_other_organisms_outcomes.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])


In [ ]:
# Notes

# - For cross-species harmonization into human, see `idtrack/docs/_notebooks/06_tutorial_humanization_mouse_pig_to_human.ipynb`.
# - This notebook intentionally stays within-species (no ortholog mapping).


## Marketing extension: export the non-human summary table

A compact CSV export is useful when drafting the Results text (or supplement/response): it captures the non-human portability story without requiring readers to interpret figures.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

if df is None or df.empty:
    print('No results DataFrame available; skipping export.')
else:
    out_csv = ctx.manuscript_tables / 'other_organisms_showcase_summary.csv'
    atomic_write_dataframe_csv(df, out_csv, index=False)
    atomic_write_dataframe_csv(df, ctx.experiment_outputs / 'tables' / out_csv.name, index=False)
    print('Wrote:', out_csv)
    df
